# سبق 11 - ایجنٹ سے ایجنٹ (A2A) پروٹوکول


## سیٹ اپ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## A2A پروٹوکول کیا ہے؟

**ایجنٹ-ٹو-ایجنٹ (A2A) پروٹوکول** ایک کھلا معیار ہے جو AI ایجنٹس کو بات چیت کرنے، 
ایک دوسرے کو دریافت کرنے، اور تعاون کرنے کے قابل بناتا ہے — چاہے وہ مختلف فریم ورکس پر مبنی ہوں یا مختلف خدمات کی میزبانی کر رہے ہوں۔


اہم تصورات:

- **دریافت** – ایجنٹس ایک *ایجنٹ کارڈ* شائع کرتے ہیں جو ان کی صلاحیتوں کی وضاحت کرتا ہے، جس سے
  دوسرے ایجنٹس (یا آرکیسٹریٹرز) کے لئے کسی کام کے لئے مناسب ماہر کو تلاش کرنا آسان ہو جاتا ہے۔
- **پیغام رسانی** – ایجنٹس ایک عام پروٹوکول کے ذریعے منظم پیغامات کا تبادلہ کرتے ہیں، تاکہ ایک
  ایجنٹ کی درخواست دوسرے ایجنٹ کی داخلی عملدرآمد سے قطع نظر سمجھی اور پوری کی جا سکے۔
  عمل۔
- **کام کی زندگی کا چکر** – A2A ایسے حالات کی تعریف کرتا ہے جیسے *جمع شدہ*، *کام میں*، *مکمل*، اور
  *ناکام*، جو آرکیسٹریٹر کو اس بات کی مکمل نظر دیتا ہے کہ ایک تفویض شدہ کام کیسے آگے بڑھ رہا ہے۔

اس سبق میں ہم A2A انداز کے تعاون کی نقل کرتے ہیں جس میں تین مخصوص سفری ایجنٹس کو ایک ورک فلو میں جوڑا جاتا ہے
جہاں ہر ایجنٹ اپنی مہارت کا حصہ ڈالتا ہے اور نتائج اگلے کو منتقل کرتا ہے۔


## مخصوص سفری ایجنٹس بنانا


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## ورک فلو کے ذریعے کثیر ایجنٹ تعاون

ہم تینوں ایجنٹس کو ایک تسلسلی ورک فلو میں جوڑتے ہیں جو A2A پیغام رسانی کی عکاسی کرتا ہے:

1. **CurrencyExchangeAgent** صارف کی درخواست وصول کرتا ہے اور کرنسی کی رہنمائی فراہم کرتا ہے۔
2. **ActivityPlannerAgent** بہتر کردہ سیاق و سباق وصول کرتا ہے اور سرگرمی کی سفارشات شامل کرتا ہے۔
3. **TravelManagerAgent** دونوں ان پٹ کو حتمی سفری بریف میں ضم کرتا ہے۔


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## پیداوار میں A2A کو سمجھنا

پیداواری ماحول میں A2A پروٹوکول طاقتور کراس سروس منظرناموں کو فعال کرتا ہے:

| قابلیت | وضاحت |
|---|---|
| **کراس فریم ورک انٹرآپ** | ایک ایجنٹ جو کسی ایک فریم ورک کے ذریعے بنایا گیا ہو، کسی دوسرے A2A کومپلائنٹ فریم ورک کے ذریعے بنائے گئے ایجنٹ کو کام تفویض کر سکتا ہے، جو حقیقی کراس آرگنائزیشن انٹرآپریبلٹی کو ممکن بناتا ہے۔ |
| **سروس کی حدود** | ایجنٹ الگ مائیکرو سروسز، کلاؤڈ ریجنز، یا یہاں تک کہ مختلف تنظیموں میں ہوسکتے ہیں جبکہ پھر بھی بغیر کسی رکاوٹ کے تعاون کر سکتے ہیں۔ |
| **متحرک دریافت** | ایک آرکیسٹریٹر رن ٹائم پر ایک ایجنٹ کارڈ رجسٹری سے پوچھ گچھ کر کے ایک مخصوص سب ٹاسک کے لیے بہترین ماہر کو تلاش کر سکتا ہے۔ |
| **اسٹریمنگ اور پش نوٹیفیکیشنز** | A2A رئیل ٹائم پیش رفت کی اپڈیٹس کے لیے سرور-سینٹ ایونٹس (SSE) اور طویل المدتی کاموں کے لیے پش نوٹیفیکیشنز کو سپورٹ کرتا ہے۔ |

جو ورک فلو ہم نے اوپر بنایا ہے وہ اس پیٹرن کا ایک سادہ، ان-پروسیس ورژن ہے۔ حقیقی
تعیناتی میں ہر ایجنٹ HTTP اینڈپوائنٹ ظاہر کرے گا، ایک ایجنٹ کارڈ شائع کرے گا، اور
A2A JSON-RPC پروٹوکول کے ذریعے بات چیت کرے گا۔


## خلاصہ

اس سبق میں آپ نے سیکھا:

1. **A2A پروٹوکول کیا ہے** — ایجنٹ سے ایجنٹ دریافت، پیغام رسانی، اور ٹاسک مینجمنٹ کے لیے ایک کھلا معیار۔
   اور ٹاسک مینجمنٹ۔
2. **خصوصی ایجنٹ کیسے بنائیں** — کرنسی ایکسچینج ایجنٹ، ایکٹوِٹی پلانر ایجنٹ،
   اور ٹریول مینیجر آرکیسٹریٹر۔
3. **ایجنٹس کو ورک فلو میں کیسے جوڑیں** — ایجنٹس کے درمیان تسلسلی پیغام رسانی ماڈل کرنے کے لیے `WorkflowBuilder` کا استعمال۔
   ایجنٹس کے درمیان تسلسلی پیغام رسانی ماڈل کرنے کے لیے `WorkflowBuilder` کا استعمال۔
4. **A2A پروڈکشن میں کیسے کام کرتا ہے** — کراس فریم ورک، کراس سروس تعاون کو متحرک دریافت اور اسٹریمنگ اپڈیٹس کے ساتھ فعال کرنا۔
4. **A2A پروڈکشن میں کیسے کام کرتا ہے** — کراس فریم ورک، کراس سروس تعاون کو متحرک دریافت اور اسٹریمنگ اپڈیٹس کے ساتھ فعال کرنا۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
